# RAG — Stage 4: Augmented generation

**Goal:** **retrieve** relevant chunks, then **generate** an answer with the LLM using only that context (same system/user prompt style as `qbrain_rag.infrastructure.llm`).

**Pipeline:** load → chunk → index → `retrieve_top_k` → `answer_with_context`.

**Requirements:** `OPENAI_API_KEY` in `rag_lab/.env`.

**Optional next:** `05_document_pipeline.ipynb` for feature extraction + test cases, or `06_evaluation_metrics.ipynb` for quantitative checks.


In [1]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

# Run this notebook from the notebooks/ folder (parent = rag_lab).
RAG_LAB = Path("..").resolve()
SRC = RAG_LAB / "src"
sys.path.insert(0, str(SRC))

from qbrain_rag.application.chunking import chunk_text
from qbrain_rag.infrastructure.vector_store import retrieve_top_k
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path
from qbrain_rag.infrastructure.document_loaders import load_document
from qbrain_rag.infrastructure.llm import answer_with_context
from qbrain_rag.infrastructure.vector_store import build_faiss_store

srs_path = resolve_default_srs_path(RAG_LAB)
text = load_document(str(srs_path))
chunks = chunk_text(text)
metadata = [{"source_file": srs_path.name, "chunk_id": i + 1} for i in range(len(chunks))]
store = build_faiss_store(chunks, metadata)

question = "What should the system display while a function is being processed?"
docs = retrieve_top_k(store, question, k=5)
answer = answer_with_context(question, docs)

print("=== Question ===\n", question)
print("\n=== Answer ===\n")
print(answer)


=== Question ===
 What should the system display while a function is being processed?

=== Answer ===

The system shall display an icon indicating the processing while a function is being processed (UL-5).
